# Images plot to understand the dataset 

In [ ]:

import keras
import matplotlib.pyplot as plt

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data(path="mnist.npz")

fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for i in range(5):
    axes[i].imshow(x_train[i], cmap='gray') 
    axes[i].set_title(f"Label: {y_train[i]}")
    axes[i].axis('off')

# 3. Render the plot
plt.show()

 # Principal Component Analysis (PCA) Implementation
 To  for Feature Extraction, we implement PCA from scratch. This allows us to reduce the 784-pixel input space into a lower-dimensional representation while preserving the maximum variance of the handwritten digits.Mathematical Steps:Mean Centering: Shift the data so the average of each pixel is 0.$$X_{centered} = X - \mu$$Covariance Matrix: Calculate how pixels vary in relation to one another.$$\Sigma = \frac{1}{n-1} X^T X$$Eigendecomposition: Find the eigenvectors (principal components) and eigenvalues (variance magnitude) of the covariance matrix.$$\Sigma v = \lambda v$$Projection: Project the original data onto the top $k$ eigenvectors to reduce dimensionality.

In [ ]:
import numpy as np

class CustomPCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None 
        self.mean = None       

    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean

        N = X.shape[0]
        covariance_matrix = np.dot(X_centered.T, X_centered) / (N - 1)

        eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
        sorted_indices = np.argsort(eigenvalues)[::-1]
    
        eigenvalues = eigenvalues[sorted_indices]
        eigenvectors = eigenvectors[:, sorted_indices]
        
        self.components = eigenvectors[:, :self.n_components]
    def transform(self, X):
        
        X_centered = X - self.mean
        
        X_reduced = np.dot(X_centered, self.components)
        return X_reduced

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)




 # Handling Class Imbalance
 Our Phase 1 "0 vs Not 0" strategy creates a significant class imbalance. To ensure our evaluation metrics (Precision, Recall, and F1-score) are not biased toward the majority class, we calculate Class Weights. These weights can be passed into the cost functions of our models to penalize the misclassification of a "0" more heavily than a "not 0".The weight for class $j$ is calculated as:$$W_j = \frac{N}{k \times n_j}$$Where $N$ is the total samples, $k$ is the number of classes, and $n_j$ is the count of samples in class $j$.

In [ ]:
def get_class_weights(y):
    
    counts = np.bincount(y) 
    n_samples = len(y)
    n_classes = len(counts)
    
    weights = n_samples / (n_classes * counts)
    
    return {0: weights[0], 1: weights[1]}


# Preprocessing 
in this section, we implement the primary data pipeline to prepare the MNIST dataset  for our machine learning models. The pipeline performs the following essential steps:
### Binary Labeling: 
 We convert the standard 10-class labels into a binary format (Class 0 for the digit "0" and Class 1 for all other digits "Not 0").
 ### No Resizing Needed:
  While the project allows for image resizing, it is unnecessary here because every single image in the MNIST dataset is already uniform at exactly 28x28 pixels.
 ### Normalization: 
 We scale the pixel intensities from integers (0–255) to floating-point values between 0.0 and 1.0 to ensure stable gradients and faster convergence.
 ### Data Splitting:
  We partition the dataset into Training, Validation, and Testing sets. We carve out 10% of the training data (6,000 samples) to serve as the validation set for hyperparameter tuning.
  ### Feature Extraction: 
  The pipeline supports both raw Flattening (784 features) and our Custom PCA implementation to reduce dimensionality.

In [ ]:
import keras
import numpy as np

def preprocess(feature_method="flatten", n_pca=50):
    print("Loading MNIST dataset...")
    # Load the raw dataset
    (X_train_full, y_train_full), (X_test, y_test) = keras.datasets.mnist.load_data(path="mnist.npz")

    # Label Conversion (0 or Not 0)
    y_train_full = np.where(y_train_full == 0, 0, 1)
    y_test = np.where(y_test == 0, 0, 1)

    #  Normalization 
    X_train_full = X_train_full.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    
   
    # Train/Validation Split 10%
    split_idx = 54000 
    X_train = X_train_full[:split_idx]
    y_train = y_train_full[:split_idx]
    
    X_val = X_train_full[split_idx:]
    y_val = y_train_full[split_idx:]
    
    print(f"Split completed: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

    weights = get_class_weights(y_train)
    #  Feature Extraction 
    if feature_method == "pca":
        X_train_flat = X_train.reshape(-1, 784)
        X_val_flat = X_val.reshape(-1, 784)
        X_test_flat = X_test.reshape(-1, 784)
        
        pca = CustomPCA(n_components=n_pca)
        
        X_train_final = pca.fit_transform(X_train_flat)
        
        X_val_final = pca.transform(X_val_flat)
        X_test_final = pca.transform(X_test_flat)
        
    elif feature_method == "flatten":
        X_train_final = X_train.reshape(-1, 784)
        X_val_final = X_val.reshape(-1, 784)
        X_test_final = X_test.reshape(-1, 784)
    
    # HOG implementation will be added here 

    return X_train_final, y_train, X_val_final, y_val, X_test_final, y_test, weights